# ConstructIA V3 — Avance de proyecto 2
## Visión por computadora e integración texto–imagen

**Objetivo:** construir un dataset académico a partir de fotografías públicas con licencia documentada en Wikimedia Commons; entrenar **ResNet50 con transferencia de aprendizaje**, evaluar **accuracy, precision, recall, F1-score y matriz de confusión**, e integrar el resultado con el clasificador de texto del Avance 2.

**Categorías visuales:** retroexcavadora (*backhoe loader*), plataforma elevadora de tijera (*scissor lift*) y minicargador (*skid-steer loader*).

> Las imágenes **no vienen incorporadas** al repositorio: se descargan desde Wikimedia Commons al ejecutar el cuaderno. El origen de cada archivo y su licencia se registran en CSV. Su pertenencia a una categoría no garantiza una etiqueta perfecta; se requiere revisión visual. No utilizar para decisiones de seguridad ni para elegir maquinaria sin validación profesional.

## 1. Preparar entorno
Abre este cuaderno en Google Colab. Selecciona **Entorno de ejecución → Cambiar tipo de entorno de ejecución → T4 GPU** si está disponible. El proyecto también puede correr en CPU, con mayor tiempo de entrenamiento. Después, **Ejecutar todas**.

El cuaderno requiere conexión a Internet para descargar las fotos autorizadas y los pesos preentrenados de ResNet50.

In [ ]:
%pip -q install requests pillow pandas scikit-learn matplotlib torch torchvision tqdm imagehash nltk
import os, sys, json, re, html, hashlib, random, time, pathlib
import numpy as np, pandas as pd
import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry
from PIL import Image, ImageOps
import imagehash
import torch, torchvision
from torchvision import transforms, models
from torch import nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report, confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

ROOT = pathlib.Path('/content/ConstructIA_V3')
IMG_DIR = ROOT / 'imagenes'
ROOT.mkdir(exist_ok=True, parents=True)
IMG_DIR.mkdir(exist_ok=True, parents=True)
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('PyTorch:', torch.__version__, '| torchvision:', torchvision.__version__, '| dispositivo:', DEVICE)

## 2. Reunir fotografías con fuente y licencia
Usaremos categorías de **Wikimedia Commons**. La meta inicial son **55 imágenes válidas por clase** (ajustable más adelante). Se guardan nombre, autor, licencia, URL de licencia y página de origen. Si hay fallos de descarga, el programa intenta con otras fotos. No atribuyas a clientes reales las fotos de terceros.

Categorías de referencia: [Backhoe loaders](https://commons.wikimedia.org/wiki/Category:Backhoe_loaders), [Scissor lifts](https://commons.wikimedia.org/wiki/Category:Scissor_lifts), [Skid-steer loaders](https://commons.wikimedia.org/wiki/Category:Skid-steer_loaders).

In [ ]:
API = 'https://commons.wikimedia.org/w/api.php'
FUENTES = {
  'retroexcavadora': ['Backhoe loaders', 'JCB backhoe loaders', 'Caterpillar backhoe loaders'],
  'plataforma_elevadora': ['Scissor lifts', 'Genie Industries scissor lifts', 'JLG scissor lifts'],
  'minicargador': ['Skid-steer loaders', 'Case skid-steer loaders', 'Caterpillar skid-steer loaders'],
}
OBJETIVO_POR_CLASE = 55
MINIMO_POR_CLASE = 24

sesion = requests.Session()
sesion.headers.update({'User-Agent': 'ConstructIA-academic-vision/0.1 (https://github.com/chrisavm24-dot/ConstructIA; classroom research)', 'Accept': 'application/json'})
retry = Retry(total=3, backoff_factor=1.5, status_forcelist=[429,500,502,503,504])
sesion.mount('https://', HTTPAdapter(max_retries=retry))

def api_imagenes(categoria):
    parametros = {
        'action':'query', 'generator':'categorymembers',
        'gcmtitle':'Category:'+categoria, 'gcmtype':'file', 'gcmlimit': '200',
        'prop':'imageinfo', 'iiprop':'url|mime|size|extmetadata',
        'iiurlwidth':'640', 'format':'json', 'formatversion':'2'
    }
    paginas=[]
    for _ in range(8):
        respuesta=sesion.get(API, params=parametros, timeout=45)
        respuesta.raise_for_status()
        objeto=respuesta.json()
        if 'error' in objeto: raise RuntimeError(f'Commons API: {objeto["error"]}')
        paginas.extend(objeto.get('query',{}).get('pages',[]))
        siguiente=objeto.get('continue')
        if not siguiente: break
        parametros.update(siguiente)
    return paginas

def meta_valor(meta, clave):
    return html.unescape(re.sub('<[^>]+>', '', meta.get(clave,{}).get('value',''))).strip()

def licencia_permitida(nombre):
    n=nombre.lower().replace('-', ' ').strip()
    return (n.startswith('cc by ') or n.startswith('cc by sa ') or n.startswith('cc0')
            or n.startswith('public domain') or n.startswith('pd '))

def candidatos_por_clase(clase):
    candidatos={}
    for categoria in FUENTES[clase]:
        try: paginas=api_imagenes(categoria)
        except requests.RequestException as exc:
            print('Problema al consultar', categoria, ':',str(exc)[:140]); continue
        for pagina in paginas:
            titulo=pagina.get('title','')
            if not titulo.startswith('File:') or titulo in candidatos: continue
            info=(pagina.get('imageinfo') or [{}])[0]
            if info.get('mime','') not in ('image/jpeg','image/png','image/webp'): continue
            meta=info.get('extmetadata') or {}
            licencia=meta_valor(meta,'LicenseShortName')
            if not licencia_permitida(licencia): continue
            url=info.get('thumburl') or info.get('url')
            if not url: continue
            candidatos[titulo]={'clase':clase,'titulo':titulo,'categoria_fuente':categoria,
                'imagen_url':url, 'pagina_fuente':info.get('descriptionurl',''),
                'autor':meta_valor(meta,'Artist')[:300], 'licencia':licencia,
                'licencia_url':meta_valor(meta,'LicenseUrl')}
        if len(candidatos)>=OBJETIVO_POR_CLASE*2: break
    items=list(candidatos.values())
    random.Random(SEED).shuffle(items)
    return items

lista_candidatos={clase:candidatos_por_clase(clase) for clase in FUENTES}
print('Fotografías candidatas con licencia:', {k:len(v) for k,v in lista_candidatos.items()})

In [ ]:
registros=[]
hashes=[]
titulos_usados=set()
for clase, candidatas in lista_candidatos.items():
    carpeta=IMG_DIR/clase; carpeta.mkdir(exist_ok=True,parents=True)
    exitos=0
    for item in tqdm(candidatas, desc='Descarga '+clase):
        if exitos >= OBJETIVO_POR_CLASE: break
        if item['titulo'] in titulos_usados: continue
        try:
            rr=sesion.get(item['imagen_url'], timeout=40)
            rr.raise_for_status()
            if len(rr.content)>8_000_000: continue
            from io import BytesIO
            foto=ImageOps.exif_transpose(Image.open(BytesIO(rr.content))).convert('RGB')
            if min(foto.size)<160: continue
            h=imagehash.phash(foto)
            # Eliminar fotografías idénticas o casi idénticas antes de separar los conjuntos
            if any(h-hprev<=3 for hprev in hashes): continue
            foto.thumbnail((768,768))
            nombre=hashlib.sha256(item['titulo'].encode('utf-8')).hexdigest()[:16]+'.jpg'
            ruta=carpeta/nombre
            foto.save(ruta,quality=85,optimize=True)
            registros.append({**item,'ruta':str(ruta),'hash_visual':str(h)})
            hashes.append(h); titulos_usados.add(item['titulo']); exitos+=1
        except (OSError, ValueError, requests.RequestException):
            continue
    print(clase,':',exitos,'imágenes descargadas')

manifest=pd.DataFrame(registros)
if manifest.empty: raise RuntimeError('No se descargaron imágenes. Revisa la conexión a Wikimedia Commons y las licencias.')
manifest.to_csv(ROOT/'origen_licencias_imagenes.csv',index=False,encoding='utf-8-sig')
print('TOTAL:',len(manifest),'| por categoría:',manifest['clase'].value_counts().to_dict())
if (manifest['clase'].value_counts() < MINIMO_POR_CLASE).any():
    raise RuntimeError('Una categoría quedó con pocas fotografías (<24). Revisa las categorías y permisos de descarga antes de entrenar.')

## 3. Revisar visualmente las etiquetas (importante)
El origen de las fotos es público, pero una categoría puede incluir material irrelevante. Inspecciona las muestras antes de entrenar. Si encuentras fotos que no pertenecen a la clase, escribe sus **títulos exactos** en `EXCLUIR_TITULOS`, ejecuta de nuevo esta celda y verifica la cantidad restante.

**No consideraremos el dataset auditado hasta que revises las miniaturas.**

In [ ]:
EXCLUIR_TITULOS = []  # Ejemplo: ['File:Nombre erróneo.jpg']
manifest=manifest[~manifest['titulo'].isin(EXCLUIR_TITULOS)].copy().reset_index(drop=True)
fig,axs=plt.subplots(3,5,figsize=(15,9))
for fila,clase in enumerate(FUENTES):
    muestras=manifest[manifest['clase']==clase].sample(min(5,(manifest['clase']==clase).sum()),random_state=42)
    for j,(_,m) in enumerate(muestras.iterrows()):
        axs[fila,j].imshow(Image.open(m['ruta']))
        axs[fila,j].set_title(m['titulo'][5:32],fontsize=8)
    for j in range(5): axs[fila,j].axis('off')
    axs[fila,0].set_ylabel(clase,fontsize=11)
plt.tight_layout(); plt.show()
print('Cantidad por categoría tras revisión:',manifest['clase'].value_counts().to_dict())
manifest.to_csv(ROOT/'origen_licencias_imagenes.csv',index=False,encoding='utf-8-sig')

## 4. Separar entrenamiento, validación y prueba
Dividimos de manera estratificada **70% / 15% / 15%** (aproximadamente). Los archivos se descargaron y deduplicaron antes de dividirse. La prueba final queda reservada hasta terminar el entrenamiento.

Para resultados académicos honestos, evita cambiar imágenes o parámetros tras consultar la prueba final; si lo haces, documenta una nueva partición independiente.

In [ ]:
train_df, temp_df = train_test_split(manifest,test_size=0.30,stratify=manifest['clase'],random_state=SEED)
val_df, test_df = train_test_split(temp_df,test_size=0.50,stratify=temp_df['clase'],random_state=SEED)
for nombre,d in [('entrenamiento',train_df),('validación',val_df),('prueba',test_df)]:
    print(nombre,':',len(d),d['clase'].value_counts().to_dict())
    d.to_csv(ROOT/f'{nombre}_manifest.csv',index=False,encoding='utf-8-sig')

## 5. ResNet50 con transferencia de aprendizaje
Usamos pesos preentrenados de ImageNet; congelamos la parte convolucional y entrenamos una nueva capa final de tres clases. Para mantener el experimento razonablemente reproducible fijamos semilla y registramos métricas de entrenamiento/validación. La red necesita fotos reales de cada clase; no genera imágenes artificiales para simular evaluación.

In [ ]:
CLASES=list(FUENTES.keys()); idx={et:i for i,et in enumerate(CLASES)}
transform_train=transforms.Compose([transforms.RandomResizedCrop(224,scale=(0.78,1.0)),transforms.RandomHorizontalFlip(),transforms.ToTensor(),
 transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])])
transform_eval=transforms.Compose([transforms.Resize(256),transforms.CenterCrop(224),transforms.ToTensor(),
 transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])])
class DatosFotos(Dataset):
    def __init__(self,df,transform): self.df=df.reset_index(drop=True);self.transform=transform
    def __len__(self): return len(self.df)
    def __getitem__(self,i):
        m=self.df.iloc[i]; foto=Image.open(m['ruta']).convert('RGB')
        return self.transform(foto), idx[m['clase']]

train_loader=DataLoader(DatosFotos(train_df,transform_train),batch_size=16,shuffle=True,num_workers=2,pin_memory=(DEVICE.type=='cuda'))
val_loader=DataLoader(DatosFotos(val_df,transform_eval),batch_size=16,shuffle=False,num_workers=2)
test_loader=DataLoader(DatosFotos(test_df,transform_eval),batch_size=16,shuffle=False,num_workers=2)
red=models.resnet50(weights=models.ResNet50_Weights.DEFAULT)
for param in red.parameters(): param.requires_grad=False
red.fc=nn.Linear(red.fc.in_features,len(CLASES))
red=red.to(DEVICE)
criterio=nn.CrossEntropyLoss(); optimizador=torch.optim.Adam(red.fc.parameters(),lr=0.001)
print('ResNet50 con ImageNet | pesos convolucionales congelados | salida de',len(CLASES),'clases')

In [ ]:
EPOCAS=5
historial=[]; mejor_f1=-1; mejor_estado=None
def evaluar_loader(loader):
    red.eval(); y_real=[];y_pred=[];perdidas=[]
    with torch.no_grad():
        for xx,yy in loader:
            xx,yy=xx.to(DEVICE), yy.to(DEVICE)
            logits=red(xx); perdidas.append(criterio(logits,yy).item())
            y_real.extend(yy.cpu().tolist()); y_pred.extend(logits.argmax(1).cpu().tolist())
    return np.mean(perdidas), np.array(y_real), np.array(y_pred)

for epoca in range(1,EPOCAS+1):
    red.eval()  # conserva las estadísticas BatchNorm del extractor congelado
    red.fc.train()
    perdidas_train=[]
    for xx,yy in train_loader:
        xx,yy=xx.to(DEVICE),yy.to(DEVICE)
        optimizador.zero_grad()
        logits=red(xx)
        perdida=criterio(logits,yy)
        perdida.backward(); optimizador.step()
        perdidas_train.append(perdida.item())
    val_loss,yv,pv=evaluar_loader(val_loader)
    _,_,f1v,_=precision_recall_fscore_support(yv,pv,average='macro',zero_division=0)
    historial.append({'epoca':epoca,'loss_entrenamiento':np.mean(perdidas_train),'loss_validacion':val_loss,
                     'f1_macro_validacion':f1v})
    if f1v>mejor_f1:
        mejor_f1=f1v; mejor_estado={k:v.detach().cpu().clone() for k,v in red.state_dict().items()}
    print(f'Época {epoca}: loss train={np.mean(perdidas_train):.3f} | loss val={val_loss:.3f} | F1 macro val={f1v:.3f}')
if mejor_estado is None: raise RuntimeError('No se obtuvo un modelo entrenado')
red.load_state_dict(mejor_estado)
torch.save({'state_dict':mejor_estado,'clases':CLASES,'imagen_fuente':'Wikimedia Commons'},ROOT/'constructia_resnet50.pt')
print('Mejor F1 de validación:',round(mejor_f1,3),'| Modelo guardado en',ROOT/'constructia_resnet50.pt')
display(pd.DataFrame(historial))

## 6. Evaluación final del modelo visual
Esta es la evaluación de **prueba** reservada. Reportamos las cinco evidencias exigidas: accuracy, precision, recall, F1-score y matriz de confusión. Las métricas son de este dataset público pequeño, **no representan precisión comercial**.

In [ ]:
test_loss, yt, yp = evaluar_loader(test_loader)
pr,rc,f1,_=precision_recall_fscore_support(yt,yp,average='macro',zero_division=0)
acc=accuracy_score(yt,yp)
print('RESULTADOS VISUALES SOBRE TEST RESERVADO:')
print('Accuracy:',round(acc,3),'| Precision macro:',round(pr,3),'| Recall macro:',round(rc,3),'| F1 macro:',round(f1,3))
print('Reporte por clase:')
print(classification_report(yt,yp,target_names=CLASES,labels=list(range(len(CLASES))),zero_division=0))
matriz=confusion_matrix(yt,yp,labels=list(range(len(CLASES))))
fig,ax=plt.subplots(figsize=(7,6))
ConfusionMatrixDisplay(matriz,display_labels=CLASES).plot(ax=ax,xticks_rotation=25,colorbar=False,cmap='Blues')
ax.set_title('ConstructIA V3 — ResNet50 (conjunto de prueba)')
plt.tight_layout();plt.show()

## 7. Primera integración texto–imagen (exigencia de la rúbrica)
Se ejecutan **ambos componentes** sobre una consulta de texto y una imagen del conjunto de prueba: el clasificador SVM predice la intención; ResNet50 predice el tipo de equipo; ConstructIA explica ambas predicciones sin afirmar disponibilidad ni precios.

Este ejemplo integra **dos clasificadores**, pero no comprueba que una fotografía corresponda a una intención de compra: es una prueba técnica de comunicación entre componentes.

In [ ]:
import subprocess
REPO='/content/ConstructIA'
if os.path.isdir(os.path.join(REPO,'.git')):
    subprocess.run(['git','-C',REPO,'pull','--ff-only'],check=True)
else:
    subprocess.run(['git','clone','https://github.com/chrisavm24-dot/ConstructIA.git',REPO],check=True)
if REPO+'/avance2' not in sys.path: sys.path.insert(0,REPO+'/avance2')
from nlp_modelo import entrenar_y_evaluar, predecir
modelo_texto=entrenar_y_evaluar()['pipeline']
consulta='Me gustaría conocer el precio para rentar una máquina'
registro_imagen=test_df.iloc[0]
texto_out=predecir(modelo_texto,consulta)
red.eval()
with torch.no_grad():
    xx=transform_eval(Image.open(registro_imagen['ruta']).convert('RGB')).unsqueeze(0).to(DEVICE)
    indice_visual=int(red(xx).argmax(1).item())
clase_visual=CLASES[indice_visual]
print('CONSULTA:',consulta)
print('OPERACIÓN 1 — TEXTO: intención detectada:',texto_out['intencion_predicha'])
print('OPERACIÓN 2 — IMAGEN: clase detectada:',clase_visual)
print('OPERACIÓN 3 — RESPUESTA DE ConstructIA:')
print(f'Entendí la intención «{texto_out["intencion_predicha"]}». La fotografía parece mostrar «{clase_visual}». ' +
      'Un asesor debe confirmar tipo de equipo, tarifa, disponibilidad y seguridad. Esta demostración no realiza reservas.')
display(Image.open(registro_imagen['ruta']).resize((420,300)))
print('Referencia de la imagen:',registro_imagen['pagina_fuente'])

## 8. Conclusiones, límites y evidencia para el documento
**Capturas recomendadas:** cantidades y muestra de imágenes / tabla con autores y licencias / división train-val-test / entrenamiento ResNet50 / métricas y matriz de confusión / respuesta integrada texto–imagen.

**Limitaciones:** las categorías de Wikimedia Commons pueden contener fotos de mala calidad o mal etiquetadas; 55 imágenes por clase son pocas para producción; la prueba es sensible a la partición y a las fuentes; la clasificación identifica una familia visual, **no confirma el modelo exacto**; no hay inventarios ni confirmación humana.

**Mejoras:** curación manual completa; más imágenes por clase y diferentes perspectivas; eliminar fotografías del mismo equipo/serie a ambos lados de la partición; calibración y clase «otro»; prueba externa con fotos de proveedores autorizados.

**Fuentes:**
- Wikimedia Commons, [Backhoe loaders](https://commons.wikimedia.org/wiki/Category:Backhoe_loaders), [Scissor lifts](https://commons.wikimedia.org/wiki/Category:Scissor_lifts), [Skid-steer loaders](https://commons.wikimedia.org/wiki/Category:Skid-steer_loaders). Cada foto conserva atribución y licencia en `origen_licencias_imagenes.csv`.
- PyTorch, documentación de [ResNet50](https://pytorch.org/vision/stable/models/generated/torchvision.models.resnet50.html) y transferencia de aprendizaje.
- Repositorio académico [ConstructIA](https://github.com/chrisavm24-dot/ConstructIA).